# Experiment B2 — Country-based validation (standalone)

Minimal, self-contained script to answer *Algorithms* reviewer #1's Comment 2
for Experiment B2 (`SME_WIN`) only: `GroupKFold` on `ISO_COUNTRY_CODE`, so
every fold's test set is composed of entire countries unseen during
training. It intentionally skips Experiment A, Gradient Boosting, the
Decision Tree baseline, temporal validation, and SHAP — everything not
needed to reproduce this one result — so it can run to completion in a
single Colab session without depending on the full pipeline notebook.

Companion result already computed for Experiment A (see
`PROCESO_MODELADO_CV_EN_perclass_100926.ipynb`, Step 7d):

```
Random Forest — Experiment A — Country-based validation (GroupKFold, k=5)
  Mean f1   0.1443 ± 0.0120   (vs. 0.3550 ± 0.0123 under k=10 stratified CV)
```

Adjust `ruta_parquet` in the first cell before running.


In [1]:
# ── COLAB SETUP ─────────────────────────────────────────────────────────
# Step 1: mount Google Drive
from google.colab import drive
import os
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

# Step 2: install libraries not included in Colab
!pip install imbalanced-learn -q

# Step 3: copy the parquet file from Drive to the local Colab environment
# ─── ADJUST this path to the folder where you uploaded the parquet in your Drive ───
DRIVE_PARQUET = '/content/drive/MyDrive/fci-datos/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
LOCAL_PARQUET = '/content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
# ────────────────────────────────────────────────────────────────────────────

if not os.path.exists(LOCAL_PARQUET):
    import shutil
    shutil.copy(DRIVE_PARQUET, LOCAL_PARQUET)
    print(f'Parquet copied to {LOCAL_PARQUET}')
else:
    print(f'Parquet already available at {LOCAL_PARQUET}')

print('Setup complete.')


Mounted at /content/drive
Parquet copied to /content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet
Setup complete.


In [2]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from imblearn.over_sampling import RandomOverSampler

# Uses LOCAL_PARQUET copied to the Colab environment by the setup cell above.
df_raw = pd.read_parquet(LOCAL_PARQUET)

for col in df_raw.select_dtypes(include='string').columns:
    df_raw[col] = df_raw[col].astype(object)
for col in ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME', 'YEAR']:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype(int)

print(f'Loaded: {df_raw.shape[0]:,} rows')


Loaded: 159,752 rows


In [3]:
df_raw['GROUP_CPV'] = df_raw['CPV'].astype(str).str[:2]
cat_cols = ['B_MULTIPLE_CAE', 'B_ON_BEHALF', 'CAE_TYPE', 'MAIN_ACTIVITY',
            'ISO_COUNTRY_CODE', 'TYPE_OF_CONTRACT', 'GROUP_CPV']
for col in cat_cols:
    df_raw[col] = df_raw[col].astype(str)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_arr = ohe.fit_transform(df_raw[cat_cols])
ohe_cols = ohe.get_feature_names_out(cat_cols)

num_cols = ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS']
df_proc = pd.concat([
    df_raw[num_cols].reset_index(drop=True),
    pd.DataFrame(ohe_arr, columns=ohe_cols)
], axis=1)

df_raw['SME_WIN'] = df_raw['B_CONTRACTOR_SME'].astype(str)\
                      .str.contains('y', case=False, na=False).astype(int)
df_proc['SME_WIN'] = df_raw['SME_WIN'].reset_index(drop=True)

used_features_B2 = [
    "B_MULTIPLE_CAE_n", "B_ON_BEHALF_n",
    "GROUP_CPV_45", "GROUP_CPV_33", "GROUP_CPV_15",
    "TYPE_OF_CONTRACT_w", "MAIN_ACTIVITY_health", "ISO_COUNTRY_CODE_si",
    "NUMBER_AWARDS", "LOTS_NUMBER", "NUMBER_OFFERS",
    "CAE_TYPE_3", "MAIN_ACTIVITY_general public\\services",
    "CAE_TYPE_4", "CAE_TYPE_5", "ISO_COUNTRY_CODE_lu"
]
X_B2 = df_proc[used_features_B2].astype(float).to_numpy()
y_B2 = df_proc['SME_WIN'].to_numpy()
paises = df_raw['ISO_COUNTRY_CODE'].reset_index(drop=True).to_numpy()

print(f'X_B2 shape: {X_B2.shape}')


X_B2 shape: (159752, 16)


In [4]:
def validacion_por_pais(X, y, paises, modelo, n_splits, nombre_modelo,
                         nombre_experimento, average='macro'):
    """
    GroupKFold on ISO_COUNTRY_CODE: every fold's test set is composed of
    entire countries absent from that fold's training set. Country group
    sizes are highly unequal (Poland alone ~40% of records), so fold sizes
    will differ substantially — this is expected and part of what is being
    tested (generalization to smaller/under-represented procurement markets).
    """
    gkf = GroupKFold(n_splits=n_splits)
    ros = RandomOverSampler(random_state=42)
    metricas = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

    print(f'\n{"="*60}')
    print(f'{nombre_modelo} — Experiment {nombre_experimento} — '
          f'Country-based validation (GroupKFold, k={n_splits})')
    print(f'{"="*60}')

    for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=paises), 1):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        paises_test = sorted(set(paises[te_idx]))

        modelo_fold = clone(modelo)
        X_tr_bal, y_tr_bal = ros.fit_resample(X_tr, y_tr)
        modelo_fold.fit(X_tr_bal, y_tr_bal)
        y_pred = modelo_fold.predict(X_te)

        acc = accuracy_score(y_te, y_pred)
        pre = precision_score(y_te, y_pred, average=average, zero_division=0)
        rec = recall_score(y_te, y_pred, average=average, zero_division=0)
        f1  = f1_score(y_te, y_pred, average=average, zero_division=0)
        for m, v in zip(['accuracy', 'precision', 'recall', 'f1'], [acc, pre, rec, f1]):
            metricas[m].append(v)

        print(f'  Fold {fold} — held-out countries: {paises_test}')
        print(f'    n_test={len(te_idx):,}  Acc={acc:.4f}  Pre={pre:.4f}  Rec={rec:.4f}  F1={f1:.4f}')

    print('-'*58)
    for m in metricas:
        print(f'  Mean {m:<10} {np.mean(metricas[m]):.4f} ± {np.std(metricas[m]):.4f}')

    return metricas


In [5]:
rf_B2_model = RandomForestClassifier(
    criterion='gini', max_features='sqrt', bootstrap=True,
    max_samples=2/3, n_estimators=100, n_jobs=-1, random_state=42
)

res_pais_B2 = validacion_por_pais(
    X_B2, y_B2, paises, rf_B2_model, n_splits=5,
    nombre_modelo='Random Forest', nombre_experimento='B2', average='binary'
)



Random Forest — Experiment B2 — Country-based validation (GroupKFold, k=5)
  Fold 1 — held-out countries: ['pl']
    n_test=63,038  Acc=0.6386  Pre=0.8762  Rec=0.6738  F1=0.7618
  Fold 2 — held-out countries: ['at', 'cz', 'lu', 'ro', 'sk']
    n_test=24,190  Acc=0.5497  Pre=0.6482  Rec=0.6819  F1=0.6646
  Fold 3 — held-out countries: ['be', 'cy', 'ee', 'fi', 'fr', 'hr', 'mk', 'mt', 'nl', 'se']
    n_test=24,152  Acc=0.6441  Pre=0.8036  Rec=0.7318  F1=0.7660
  Fold 4 — held-out countries: ['bg', 'gr', 'hu', 'it', 'lt', 'no', 'pt']
    n_test=24,214  Acc=0.6621  Pre=0.8846  Rec=0.7044  F1=0.7843
  Fold 5 — held-out countries: ['de', 'dk', 'es', 'ie', 'lv', 'si', 'uk']
    n_test=24,158  Acc=0.6383  Pre=0.8581  Rec=0.6867  F1=0.7629
----------------------------------------------------------
  Mean accuracy   0.6266 ± 0.0394
  Mean precision  0.8141 ± 0.0876
  Mean recall     0.6957 ± 0.0206
  Mean f1         0.7479 ± 0.0424
